In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn import tree
from sklearn import metrics


In [ ]:
# Link in [1]
diabetes = pd.read_csv("diabetes.csv")
diabetes.tail()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1
767,1,93,70,31,0,30.4,0.315,23,0


# Data Pre-processing

In [ ]:
diabetes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


**Observation**: All the data columns are clean without the missing values

In [ ]:
features = diabetes.iloc[:,0:8]
target = diabetes.iloc[:,-1]


In [ ]:
features.tail()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
763,10,101,76,48,180,32.9,0.171,63
764,2,122,70,27,0,36.8,0.340,27
765,5,121,72,23,112,26.2,0.245,30
766,1,126,60,0,0,30.1,0.349,47
767,1,93,70,31,0,30.4,0.315,23


In [ ]:
target .tail()

,Outcome
763,0
764,0
765,0
766,1
767,0


# Feature Selection

In [ ]:
best_features = SelectKBest(score_func = chi2, k = 8)
fit = best_features.fit(features, target)
dfscores = pd.DataFrame(fit.scores_)
dfcolumns = pd.DataFrame(features.columns)
featureScores = pd.concat([dfcolumns,dfscores],axis=1)
featureScores.columns = ['Specs','Score']
print(featureScores.nlargest(10,'Score'))

                      Specs        Score
4                   Insulin  2175.565273
1                   Glucose  1411.887041
7                       Age   181.303689
5                       BMI   127.669343
0               Pregnancies   111.519691
3             SkinThickness    53.108040
2             BloodPressure    17.605373
6  DiabetesPedigreeFunction     5.392682


**Observation**: The top 3 features are Insulin, Glucose, and Age. We see that the top 2 are in the same scale of 4 digits whereas there is a different scale between Glucose 4 digits and Age 3 digits. If we just take the top 2 with the same scale, we will have so few features that there might be a risk of underfitting. Thus, we also accommodate the Age. Since BMI and Pregnancies are also on the same scale as Age, we also choose them. However, note that the majority of literature works on all the feature variables of this PIMA. Additionally, the last three variables still have scores above 0 which means that they also have a significant impact. Thus, we also choose the last three features to synchronize the result with the literature </br>
We end up choosing all 8 features: Insulin, Glucose, Age, BMI, Pregnancies, SkinThickness, BloodPressure, and  DiabetesPedigreeFunction.

In [ ]:
selected_features_name =  featureScores.nlargest(8,'Score').iloc[:8]["Specs"].values
selected_features = features[selected_features_name]
selected_features.tail()

,Insulin,Glucose,Age,BMI,Pregnancies,SkinThickness,BloodPressure,DiabetesPedigreeFunction
763,180,101,63,32.9,10,48,76,0.171
764,0,122,27,36.8,2,27,70,0.340
765,112,121,30,26.2,5,23,72,0.245
766,0,126,47,30.1,1,0,60,0.349
767,0,93,23,30.4,1,31,70,0.315


# Data Normalization

In [ ]:
selected_features.describe()

,Insulin,Glucose,Age,BMI,Pregnancies,SkinThickness,BloodPressure,DiabetesPedigreeFunction
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,79.799479,120.894531,33.240885,31.992578,3.845052,20.536458,69.105469,0.471876
std,115.244002,31.972618,11.760232,7.884160,3.369578,15.952218,19.355807,0.331329
min,0.000000,0.000000,21.000000,0.000000,0.000000,0.000000,0.000000,0.078000
25%,0.000000,99.000000,24.000000,27.300000,1.000000,0.000000,62.000000,0.243750
50%,30.500000,117.000000,29.000000,32.000000,3.000000,23.000000,72.000000,0.372500
75%,127.250000,140.250000,41.000000,36.600000,6.000000,32.000000,80.000000,0.626250
max,846.000000,199.000000,81.000000,67.100000,17.000000,99.000000,122.000000,2.420000


**Observation**: We can observe that there is a huge gap between the two largest max value of *Insulin* and *Glucose* which is 846/199 = 4.26. The gap is even larger between *Insulin* and *Pregnancies* which is 846/17 = 49.76. This difference in maximum value could lead the model bias more toward the larger scale even though all of them have equal importance. Thus, those selected features need to be normalized. We used the Min-Max Scaler to rescale all of them into the range of 0 and 1

In [ ]:
normalized_features = selected_features.copy()
normalized_features[selected_features_name] = MinMaxScaler().fit_transform(selected_features[selected_features_name])
normalized_features.tail()

,Insulin,Glucose,Age,BMI,Pregnancies,SkinThickness,BloodPressure,DiabetesPedigreeFunction
763,0.212766,0.507538,0.700000,0.490313,0.588235,0.484848,0.622951,0.039710
764,0.000000,0.613065,0.100000,0.548435,0.117647,0.272727,0.573770,0.111870
765,0.132388,0.608040,0.150000,0.390462,0.294118,0.232323,0.590164,0.071307
766,0.000000,0.633166,0.433333,0.448584,0.058824,0.000000,0.491803,0.115713
767,0.000000,0.467337,0.033333,0.453055,0.058824,0.313131,0.573770,0.101196


**Observation**: We now can see the clear impact of each features. The data now is ready for training

# Model building

#### Splitting dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
                                    normalized_features,
                                    target,
                                    test_size=0.1,
                                    random_state=0)
num_model = 10

#### SVM

In [ ]:

"""
1. kernel='rbf': the study make no assumption about the input data so it could
be either lienarly seperable or non-linearly seperable. Thus, rbf is chosen to
handle unseen and unexpected input
2. C = 1: smaller C parameter could make the model have a better generalization
for unseen data so 1 is chosen for generalization
3. random_state = 0: this is the random seed for future study if they want to
replciate the result
"""

base_SVM_clf = SVC(kernel='rbf', C=1.0, random_state=0)
bag_SVM_clf = BaggingClassifier(
                estimator = base_SVM_clf,
                n_estimators = num_model,
                random_state = 0)
bag_SVM_clf.fit(X_train,y_train)


BaggingClassifier(estimator=SVC(random_state=0), random_state=0)

#### Base Decision Tree

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
dt_param_grid = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [10, 20, 40]
}

#using grid search we can find sweet spot in max_depth to prevent from overfitting
grid_search_dt = GridSearchCV(
    estimator= tree.DecisionTreeClassifier(random_state=0),
    param_grid=dt_param_grid,
    cv=5,
    scoring='accuracy'
)
grid_search_dt.fit(X_train, y_train)

base_DT_clf = grid_search_dt.best_estimator_
print(f"Best Parameters Found: {grid_search_dt.best_params_}")

bag_DT_clf = BaggingClassifier(
                estimator = base_DT_clf,
                n_estimators = num_model,
                random_state = 0)

bag_DT_clf.fit(X_train, y_train)

Best Parameters Found: {'max_depth': 5, 'min_samples_split': 20}


BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=5,
                                                   min_samples_split=20,
                                                   random_state=0),
                  random_state=0)

# Evaluation

#### SVM Result

In [ ]:
print("Train - Accuracy :", metrics.accuracy_score(y_train, bag_SVM_clf.predict(X_train)))
print("Train - Confusion matrix :\n",metrics.confusion_matrix(y_train, bag_SVM_clf.predict(X_train)))
print("Train - classification report :\n", metrics.classification_report(y_train, bag_SVM_clf.predict(X_train)))
print("")
print("---------------------------------------------------------------------------------")
print("")
print("Test - Accuracy :", metrics.accuracy_score(y_test, bag_SVM_clf.predict(X_test)))
print("Test - Confusion matrix :\n",metrics.confusion_matrix(y_test, bag_SVM_clf.predict(X_test)))
print("Test - classification report :\n", metrics.classification_report(y_test, bag_SVM_clf.predict(X_test)))

Train - Accuracy : 0.7930535455861071
Train - Confusion matrix :
 [[407  42]
 [101 141]]
Train - classification report :
               precision    recall  f1-score   support

           0       0.80      0.91      0.85       449
           1       0.77      0.58      0.66       242

    accuracy                           0.79       691
   macro avg       0.79      0.74      0.76       691
weighted avg       0.79      0.79      0.79       691


---------------------------------------------------------------------------------

Test - Accuracy : 0.8441558441558441
Test - Confusion matrix :
 [[46  5]
 [ 7 19]]
Test - classification report :
               precision    recall  f1-score   support

           0       0.87      0.90      0.88        51
           1       0.79      0.73      0.76        26

    accuracy                           0.84        77
   macro avg       0.83      0.82      0.82        77
weighted avg       0.84      0.84      0.84        77



#### Decision Tree Result

In [ ]:
print("Train - Accuracy :", metrics.accuracy_score(y_train, bag_DT_clf.predict(X_train)))
print("Train - Confusion matrix :\n",metrics.confusion_matrix(y_train, bag_DT_clf.predict(X_train)))
print("Train - classification report :\n", metrics.classification_report(y_train, bag_DT_clf.predict(X_train)))
print("")
print("---------------------------------------------------------------------------------")
print("")
print("Test - Accuracy :", metrics.accuracy_score(y_test, bag_DT_clf.predict(X_test)))
print("Test - Confusion matrix :\n",metrics.confusion_matrix(y_test, bag_DT_clf.predict(X_test)))
print("Test - classification report :\n", metrics.classification_report(y_test, bag_DT_clf.predict(X_test)))

Train - Accuracy : 0.8408104196816208
Train - Confusion matrix :
 [[396  53]
 [ 57 185]]
Train - classification report :
               precision    recall  f1-score   support

           0       0.87      0.88      0.88       449
           1       0.78      0.76      0.77       242

    accuracy                           0.84       691
   macro avg       0.83      0.82      0.82       691
weighted avg       0.84      0.84      0.84       691


---------------------------------------------------------------------------------

Test - Accuracy : 0.7922077922077922
Test - Confusion matrix :
 [[43  8]
 [ 8 18]]
Test - classification report :
               precision    recall  f1-score   support

           0       0.84      0.84      0.84        51
           1       0.69      0.69      0.69        26

    accuracy                           0.79        77
   macro avg       0.77      0.77      0.77        77
weighted avg       0.79      0.79      0.79        77



**Observation**:
<br/> Firstly, we can see that there is significant change in the accuracy performance of both models
- SVM: in the litterature they have 83% whereas our study found 85%
- Decision Tree: In the litterature they have around 74.6% but our study found 79%
<br/> Secondly, the bagging SVM  tend to perform better than the bagging Decision Tree not only in accuracy but also accross other metrics such as precision, recall, and F1 score for both diabetes and not-diabetes.<br/>

# Reference
[1] Pima Indians Diabetes Database, Kaggle, n.d. [Online]. Available: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database/data